# 🎮 Day 02a: Command & State Patterns

---

## 🎯 What You'll Master Today
- **Command**: encapsulate a request as an object (undo/redo, queues, macros)
- **State**: change behavior when internal state changes (state machine)
- Real-world: text editors, vending machines, order workflows

---

## 🎭 The Analogies

**Command** = Restaurant order slip  
The waiter writes your order on a slip (command object). The kitchen executes it. The slip can be undone ("cancel that order"), queued, or logged.

**State** = Traffic light  
Red → Green → Yellow → Red. The light behaves differently in each state (cars stop/go/slow), and it transitions automatically. Same object, different behavior.

---

In [ ]:
# ============================================
# 1. COMMAND — Text Editor with Undo/Redo
# ============================================
from abc import ABC, abstractmethod

class Command(ABC):
    @abstractmethod
    def execute(self) -> None: pass
    @abstractmethod
    def undo(self) -> None: pass

class TextEditor:
    """Receiver — the actual object being modified."""
    def __init__(self):
        self.content = ""
    
    def insert(self, text: str, pos: int):
        self.content = self.content[:pos] + text + self.content[pos:]
    
    def delete(self, pos: int, length: int) -> str:
        deleted = self.content[pos:pos + length]
        self.content = self.content[:pos] + self.content[pos + length:]
        return deleted


class InsertCommand(Command):
    def __init__(self, editor: TextEditor, text: str, pos: int):
        self._editor = editor
        self._text = text
        self._pos = pos
    
    def execute(self):
        self._editor.insert(self._text, self._pos)
    
    def undo(self):
        self._editor.delete(self._pos, len(self._text))


class DeleteCommand(Command):
    def __init__(self, editor: TextEditor, pos: int, length: int):
        self._editor = editor
        self._pos = pos
        self._length = length
        self._deleted = ""  # stored for undo
    
    def execute(self):
        self._deleted = self._editor.delete(self._pos, self._length)
    
    def undo(self):
        self._editor.insert(self._deleted, self._pos)


class CommandHistory:
    """Invoker — manages command execution and undo/redo."""
    def __init__(self):
        self._history: list[Command] = []
        self._redo_stack: list[Command] = []
    
    def execute(self, cmd: Command):
        cmd.execute()
        self._history.append(cmd)
        self._redo_stack.clear()  # new action clears redo
    
    def undo(self):
        if self._history:
            cmd = self._history.pop()
            cmd.undo()
            self._redo_stack.append(cmd)
    
    def redo(self):
        if self._redo_stack:
            cmd = self._redo_stack.pop()
            cmd.execute()
            self._history.append(cmd)


# Demo
editor = TextEditor()
history = CommandHistory()

history.execute(InsertCommand(editor, "Hello ", 0))
print(f"After insert: '{editor.content}'")

history.execute(InsertCommand(editor, "World!", 6))
print(f"After insert: '{editor.content}'")

history.undo()
print(f"After undo:   '{editor.content}'")

history.undo()
print(f"After undo:   '{editor.content}'")

history.redo()
print(f"After redo:   '{editor.content}'")

In [ ]:
# ============================================
# 2. STATE — Order Processing Workflow
# ============================================
from abc import ABC, abstractmethod

class OrderState(ABC):
    @abstractmethod
    def process(self, order: 'Order') -> str: pass
    @abstractmethod
    def cancel(self, order: 'Order') -> str: pass
    @abstractmethod
    def status(self) -> str: pass

class PendingState(OrderState):
    def process(self, order):
        order._state = PaidState()
        return "✅ Payment received. Order is now PAID."
    def cancel(self, order):
        order._state = CancelledState()
        return "❌ Order cancelled."
    def status(self): return "⏳ PENDING"

class PaidState(OrderState):
    def process(self, order):
        order._state = ShippedState()
        return "📦 Order shipped!"
    def cancel(self, order):
        order._state = CancelledState()
        return "💸 Refund initiated. Order cancelled."
    def status(self): return "💰 PAID"

class ShippedState(OrderState):
    def process(self, order):
        order._state = DeliveredState()
        return "🏠 Order delivered!"
    def cancel(self, order):
        return "🚫 Cannot cancel — already shipped!"
    def status(self): return "🚚 SHIPPED"

class DeliveredState(OrderState):
    def process(self, order):
        return "✅ Order already delivered."
    def cancel(self, order):
        return "🚫 Cannot cancel — already delivered!"
    def status(self): return "✅ DELIVERED"

class CancelledState(OrderState):
    def process(self, order):
        return "🚫 Cannot process — order is cancelled."
    def cancel(self, order):
        return "Already cancelled."
    def status(self): return "❌ CANCELLED"


class Order:
    def __init__(self, order_id: int):
        self.order_id = order_id
        self._state: OrderState = PendingState()
    
    def process(self) -> str:
        return self._state.process(self)
    
    def cancel(self) -> str:
        return self._state.cancel(self)
    
    def status(self) -> str:
        return f"Order #{self.order_id}: {self._state.status()}"


# Walkthrough
order = Order(101)
print(order.status())
print(order.process())   # pending → paid
print(order.status())
print(order.process())   # paid → shipped
print(order.status())
print(order.cancel())    # can't cancel shipped!
print(order.process())   # shipped → delivered
print(order.status())

```
╔══════════════════════════════════════════════════════════╗
║              ORDER STATE MACHINE                         ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║   PENDING ──process()──→ PAID ──process()──→ SHIPPED    ║
║     │                     │                    │         ║
║   cancel()             cancel()            process()    ║
║     ↓                     ↓                    ↓         ║
║  CANCELLED            CANCELLED            DELIVERED     ║
║                                                          ║
╚══════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | What is the Command pattern? | Encapsulates a request as an object. Enables undo/redo, queuing, logging |
| 2 | Command vs Strategy? | Command wraps a specific request with undo. Strategy swaps algorithms without undo |
| 3 | What is the State pattern? | Object changes behavior when internal state changes. Eliminates complex if/elif chains |
| 4 | State vs Strategy? | State: object transitions between states automatically. Strategy: client picks the algorithm |
| 5 | Real-world Command examples? | Ctrl+Z undo, database transactions, task queues (Celery), GUI button actions |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Command = request as object (execute + undo)          │
## │  • State = behavior changes with state transitions       │
## │  • Command enables: undo/redo, macro, queue, log         │
## │  • State eliminates giant if/elif state checks            │
## │  • Both use polymorphism to avoid conditionals            │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Template Method & Iterator** — define algorithm skeletons and traverse collections